In [1]:
import dotenv
import os
from dotenv import load_dotenv
load_dotenv()
if os.environ["GEMINI_API_KEY"]:
    print("API KEY IS SET")

API KEY IS SET


In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
print("ChatGoogleGenerativeAI initialized successfully!")

ChatGoogleGenerativeAI initialized successfully!


In [3]:
response = llm.invoke("What is AI? Tell me in one line.")
print(response.text)

AI is the simulation of human intelligence by machines, enabling them to learn, reason, and solve problems.


# RAG IMPLEMENTATION WITH YOUR OWN TEXT DATA

### STEP:1 Preparing Document for your text

In [5]:
from langchain_core.documents import Document

## Your text data
my_text="""Claude is a series of large language models (LLMs) developed by American software company Anthropic. Claude was released as an AI-based chatbot in March 2023. It is also used in AI-assisted software development. Anthropic sells Claude-based agentic tools including Claude Code, a terminal coding agent, and Claude Duet, which runs in a virtual machine, allowing it to run without input from a user.Claude is trained using a constitution, a technique developed by Anthropic to improve ethical and legal compliance. Since Claude 3, each generation has typically been released in three sizes, named Haiku (the least capable), Sonnet, and Opus (the most capable). In 2026, a more capable model named Claude Mythos was released to a handful of companies and organizations; this was followed by the release of Claude Fable, a version of Mythos with stricter safeguards, to the general public.In February 2026, the United States Department of Defense designated Anthropic a "supply chain risk" and federal agencies were ordered to phase out Claude after the company refused to remove contractual prohibitions on using Claude for mass domestic surveillance and fully autonomous weapons. A federal judge blocked the designation in March 2026 and permanently set it aside in August 2026 as unconstitutional retaliation."""
docs=[Document(page_content=my_text,metadata={"source":"ABS","documentID":"s1"})]
docs

[Document(metadata={'source': 'ABS', 'documentID': 's1'}, page_content='Claude is a series of large language models (LLMs) developed by American software company Anthropic. Claude was released as an AI-based chatbot in March 2023. It is also used in AI-assisted software development. Anthropic sells Claude-based agentic tools including Claude Code, a terminal coding agent, and Claude Duet, which runs in a virtual machine, allowing it to run without input from a user.Claude is trained using a constitution, a technique developed by Anthropic to improve ethical and legal compliance. Since Claude 3, each generation has typically been released in three sizes, named Haiku (the least capable), Sonnet, and Opus (the most capable). In 2026, a more capable model named Claude Mythos was released to a handful of companies and organizations; this was followed by the release of Claude Fable, a version of Mythos with stricter safeguards, to the general public.In February 2026, the United States Depart

#### STEP:2 Splitting the Document into CHUNKS

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
split=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks=split.split_documents(docs)
chunks

[Document(metadata={'source': 'ABS', 'documentID': 's1'}, page_content='Claude is a series of large language models (LLMs) developed by American software company Anthropic. Claude was released as an AI-based chatbot in March 2023. It is also used in AI-assisted software development. Anthropic sells Claude-based agentic tools including Claude Code, a terminal coding agent, and Claude Duet, which runs in a virtual machine, allowing it to run without input from a user.Claude is trained using a constitution, a technique developed by Anthropic to improve ethical and'),
 Document(metadata={'source': 'ABS', 'documentID': 's1'}, page_content='developed by Anthropic to improve ethical and legal compliance. Since Claude 3, each generation has typically been released in three sizes, named Haiku (the least capable), Sonnet, and Opus (the most capable). In 2026, a more capable model named Claude Mythos was released to a handful of companies and organizations; this was followed by the release of Cla

### STEP 3: Creating Embeddings for the chunks

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embedding_model=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

In [8]:
embedding_model.embed_query("What is AI?")

[-0.012995635,
 0.0062429244,
 -0.0060949307,
 0.0041634077,
 0.0026599132,
 0.0026361085,
 -0.013465155,
 0.02526141,
 -0.0029722652,
 -0.05672356,
 -0.006702272,
 -0.003428023,
 -0.04120243,
 -0.0092308195,
 0.016912486,
 -0.0040361793,
 0.031953327,
 0.00152218,
 -0.00035570224,
 0.0018425352,
 -0.014674235,
 0.0047656354,
 0.012941361,
 0.021710685,
 -0.008286402,
 0.028978454,
 0.0048153205,
 -0.0068676965,
 -0.0124633005,
 0.13349202,
 -0.023295544,
 -0.022616822,
 0.008208155,
 0.005089146,
 -0.0070315646,
 0.0047773747,
 -0.011023618,
 -0.025647925,
 0.0027397184,
 2.3624034e-05,
 -1.8208999e-05,
 0.016080216,
 0.017973324,
 0.0064955927,
 -0.004011068,
 -0.0049498905,
 -0.03247996,
 -0.013147978,
 0.008291769,
 -0.0011112508,
 0.0009616394,
 0.022400528,
 -0.02360695,
 -0.04574031,
 0.01399749,
 -0.0026479582,
 0.019530054,
 -0.007001244,
 0.008359673,
 -0.001310887,
 -0.026255924,
 0.02350381,
 -0.0063299,
 0.03151435,
 -0.0024238557,
 0.003936887,
 0.006393735,
 -0.03946539,

### STEP 4: Create and store  Embeddings in the Vector Store 

In [15]:
from langchain_chroma import Chroma
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [16]:
vectors=[]
for docs in chunks:
    vector=embedding_model.embed_documents([docs.page_content])
    vectors.append(vector)

### STEP 5: Semantic Search

In [17]:
context=vectorstore.similarity_search("What is Claude?",k=3)

## Talk to llm

In [18]:
response=llm.invoke(f"What is ClaudeAI? you can answer using the following context:{context}")
print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, **Claude** is a series of large language models (LLMs) developed by the American software company Anthropic. \n\nKey details about Claude include:\n* **Release & Use:** It was released as an AI-based chatbot in March 2023 and is also used in AI-assisted software development.\n* **Training:** It is trained using a "constitution"—a technique developed by Anthropic to improve ethical and legal compliance.\n* **Model Sizes & Generations:** Since Claude 3, generations have typically been released in three sizes: Haiku (least capable), Sonnet, and Opus (most capable). In 2026, newer models were introduced, including Claude Mythos (released to a select few) and Claude Fable (a version of Mythos with stricter safeguards released to the general public).\n* **Agentic Tools:** Anthropic also sells Claude-based agentic tools, such as *Claude Code* (a terminal coding agent) and *Claude Duet* (which runs in a virtual machine without needing u